```raw
n - description number of satisfactory machine
y = M(n) - y is result of this satisfactory machine
a_n = F(M(n)) where F is some function from computable number (y) to computable number (a_n)

phi(n) = n' - where n is description number of satisfactory machine and n' is also a dnsm (but can have modifications)

for instance if n desribes a machine that compute number x
then n' can describe 2 * x + 1

then f(a_n) = a_(phi(n)) = F(M(phi(n)))
```

In [11]:
from math import *

def problematic_sequence():
    n = 0
    while True:
        # Irregular oscillation that eventually damps down
        if n % 10 == 0:  # Every 100 terms, we get a "bump"
            term = 1/(n+1)
        else:
            term = 1/((n+1)**2)
        
        # Alternating signs make it tricky
        term = term * (-1)**n
        n += 1
        yield term

def N(e):
    n = 0
    gen = problematic_sequence()
    current = next(gen)
    next_ = next(gen)
    while abs(current - next_) >= e:
        n += 1
        current = next_
        next_ = next(gen)
    return n

e = 0.01
n = N(e)

print(f"N = {n}")
validation = []
for i in range(1 << 12):
    x = sum([x for _, x in zip(range(n), problematic_sequence())])
    y = sum([x for _, x in zip(range(n + 1), problematic_sequence())])
    if abs(x - y) >= e:
        print(f"False => {n}")
        break
    n += 1



N = 13
False => 20


### Newton's method

In [70]:
#     # f(x) = x² - 2 = 0
def f(x): return x**2 - 2
def newton(epsilon):
    x = 1
    while abs(f(x)) >= epsilon:
        y0 = f(x)
        x -= y0 / ((y0 + f(x + epsilon)) / epsilon)
    return x

x = (-newton(0.000001))
s2 = (sqrt(2))
print(abs(x - s2) < 0.000001)


True


### Turing Machine for sin
```raw
sin(x) = x - x^3/3! + x^5/5! - x^7/7! + ...

sin can be calculated by this power series

The infinite loop problem:
- it doesn't practical to compute infinite power series. so we need to introduce desired precision "e"
- we will compute sin(x) up to desired precision "e"

reals are infinte problem:
- each term of this series are reals. all reals are infinite, 0.5 is just 0.5(0)
- the further terms may be just transcedentals and etc
- to compute sin(x) up to e, we need to compute N terms one by one
- but if each term is infinite, how can we go to the next term?
- to approach this we can compute sin(x) digit by digit:
```

In [88]:
from math import *

def sin_terms_gen(x):
    i = 1
    sign = True
    while True:
        if i % 2 != 0:
            yield (1 if sign else -1) * (x**i) / factorial(i)
            sign = not sign
        i += 1

def sinn(x, n):
    gen = sin_terms_gen(x)
    p = 10**n
    out = floor(next(gen) * p)
    while True:
        # the terms itself should compute to first nth non zero digit
        # but its not obvious to implement that easily
        new_out = out + floor((next(gen)) * p)

        # calculate terms until they did change nth digit
        # if no changes (terms are too small) then exit
        if out == new_out:
            break
        out = new_out
    return out / p

sinn(0.5, 10)


0.4794255383

```raw
input is also real number:
- we somehow need to provide this x to the Turing Machine which computes sin(x)
- it can be done in many ways
- but in eash of solutions we have a problem since this input number x is also real
- or even if we use radins for x, then x is transcendental. for example pi / 8
- if we write this number at the beginning of the tape, that it will take the whole tape. number x is infinite
- we have machine S which gets N first digits of the x and calculate first N digits of sin(x)
- machine X calculates the x and passes N digits to S
- so machine S and X work in tandem: begin -> machine X iteration -> machine S iteration -> begin






```

```raw
If there exists computable sequence Y which have infinite amount of 0s in it. Then there is exists E(Y, n) which return number of 1s between nth 0 and n+1 0. And then we can define phi(n) = E(Y, n) and we state that phi is computable function.

This is the definition of computable function from the computable sequence.

But we can define computable sequence from the computable function. It's showed in the code below:
```

In [95]:
def phi(n):
    return 2 * n + 1

def H(x, y):
    return phi(x) == y

def A(n, m):
    return H(n, m)

def B(n, m):
    return not H(n, m)

def K():
    n = 0
    m = 0
    while True:
        yield 0
        while True:
            if B(n, m):
                yield 1
                m += 1
            elif A(n, m):
                n += 1
                m = 0
                break

print(''.join([str(k) for _, k in zip(range(64), K())]))

0101110111110111111101111111110111111111110111111111111101111111


```raw
this is an illustration of computable funciton of integral argument which defines computable sequence.

it could be done vice versa (like Turing originally stated), where first we have a computable sequence and from that sequence the c.function is defined.

but what about computable function of a real argument? there's no such function since there is no general way to describe a real number

but! we can define computable function of computable argument!

this can be done by creating a "pipe" between 2 machines A and B. A outputs the computable argument. B takes the computable argument and outputs result of a computable function. this is called "tandem"

for example:
- B calculates pi/8
- A calculates tan of pi/8

if n is the description number of machine B and n is satisfactory,
then B calculates y_n in range [0, 1]
then machine A calculate a_n = tan(pi * (y_n - 0.5))

furthermore, if we enumerate through all satisfactory values of n (all values in range [0, 1])
then A will calculate all computable numbers in range (-inf, +inf) since tan range is (-inf, +inf)
```

### More theorems:
1. Computable function F with argument X is computable when X is computable function or computable argument
2. If F is defined recursively in terms of computable function and its argument is integral, then this F is computable
3. F(m, n) if m and n are inegers is computable
4. if phi(n) is computable and its output is 0 or 1, then it can define a computable sequence
5. Dedekind's theorem application for computable numbers:
```raw
Dedekind's cut is used for defining irrational number by rational numbers:
- to define sqrt(2) using rational numbers we divide number line into 2 sets at the sqrt(2)
- A: q < 0 OR q^2 < 2
- B: q > 0 AND q^2 > 2
- every rational number is in A or B
- any(A) < any(B)
- A + B contain every number except sqrt(2)
- sqrt(2) is not in A and not in B

Using this cut we can define irrational numbers. If the number x is not in A and not in B, then there is a CUT, otherwise there is NO CUT.

For example we can find a CUT for rational numbers (example above).

But we can't find a CUT for real numbers. For real numbers sqrt(2) will be in A or B.

For computable numbers we can also find a CUT: cut a not computable number.

But if we restrict the Dedekind cut strictly to computable numbers, than we can not find CUT for computable numbers.
```

the proofs are too rigid and mathematicaly tight... they are relly hard to grasp if you read this book as recreational activity. I've already read this chapter twice, got more understanding for second read, but it's still too hard...


### The halting problem

### Recap before Major Proof

##### Section 8. The Halting Problem

Lets assume the TM D which takes Description Number of another machine and outputs 1 if this is a circle-free machine or circular machine.

For example, here is the description of circle-free machine:
```raw
q1 * {P1, R} q1
```
so it just prints 1s idefinetely

here is a circular machine:
```raw
q1 * {R} q2
q2 * {L} q1
```
it just goes to left and right and does nothing. doesn't print indefinetely

then we have a machine U, which can take a Description Number and run the machine.

lets create a machine, that iterate over all description numbers (from 1 to inf):
- n = 0
- this description number is passed to D
- if sd is circle-free, then n++ and run sd via U for n iterations
- else continue
- n is the counter of circle-free machines and iteration bounds for U, since circle-free machines can compute indefinetely

the whole machine is H, it contains D and U

for example it does process N - 1 machines and for example all of them are circle free

it then need to process Nth description number and Nth description number is the description number of H itself

so we're stuck in recursion. so H is a contradiction to itself and hence D cannot be constructed.

That is the Halting Problem!

there can be no machine E which when supplied with DN tells whether the DN machine (M) ever prints 0

if such machine E exists, then there is a general process of determing whether DN machine (M) prints 0 infinitely often:
- let M1 machine prints the same sequence as M machine except for first 0 it prints 0'
- let M2 machine prints the same sequence as M machine except for first two 0s it prints 0'
- let M...
- let K machine will output DN of M1, M2, ..., MN (such machine exists and its easy to show)
- if M doesn't print 0, then Ms... also doesn't print 0
- if M print 0 once, then Ms... also doesn't print 0, but M1 prints 0'
- ...
- now lets combine machine E and K into G
- G goes through all Ms...
- current M machine is tested with E. if M machine never prints 0, then G prints 0, otherwise none
- this process is repeated for all further M machines
- so if M never prints 0 or prints 0 n times, then G will print 0 infinitely often
- if M prints 0 infinitely often, then G will never print 0
- since E exists, we can test G with E
- if G never prints 0, then M prints 0 infinitely often <=> M is circle-free, halting problem is solved, machine D exists!
- if G ever prints 0, then M is circular
- but previous proof showed that there is no such machine D

> so the proof of the original statement is done by rephrasing the problem into the halting problem which is prooved to be unsolvable!

### The extent of the computable numbers
In this part Turing wants to show that Turing Machine can actually compute everything that is computable.

For this he bring 3 arguments:
1. a direct appeal to intution. basically showing that Turing Machine works on the same basic principles as "Human Machine". the brief summary: everything that is computable by human, is computable by Turing Machine
2. a proof of the equivalence of two definitions.
3. giving examples of large classes of numbers which are computable. showing direct examples of computing complicated numbers such as transcendental pi and e, algebraic numbers defined via polynomial equations, power series and etc

##### Section 9. The extent of the computable number. A direct appeal to intuition
In this section Turing basically describes his view on "human computer". What it takes to do computations manually on a piece of paper. The human computer has a 2D piece of paper but it can be reduced to just a 1d tape. That human computer have states of mind which are finite. That tape cells have recognisable symbols and human computer can alter them. Also it can move attention to nearest symbols which are in range of some number L (neares symbols). Basically describing the Turing machine...


##### Section 10 
##### Propositional Logic
First we deive into propositional logic. Propositional logic operates with propositions. Each proposition is either True or False.

There are binary operations which can combine propositions such as AND, OR, Implication

There are unary operations such as NOT

Here are some of my reasonings to make sense of Implication:
```raw
если я буду хорошо учиться, то я сдам экзамен
либо я буду плохо учиться, либо я сдам экзамен

хорошо учиться | сдам экзамен | 
0              | 0            | 1
0              | 1            | 1
1              | 0            | 0
1              | 1            | 1

если я сложу 2 и 2, то получу 4
A | B | 
0 | 0 | 1
0 | 1 | 1
1 | 0 | 0
1 | 1 | 1

если человек, то смертен
человек | смертен
0 0 1
0 1 1
1 0 0
1 1 1
```

If we restrict ourself only to propositional logic, then each theorem in proposition logic is decidable:
- we have a set of propositional logic axioms
- we can form complex theorems from it
- even for hardest theorems there is a finite and general algorithm to determine validity and satisfiability

truth tables are good for that, but if we want to use truth tables for theorem with 100 propositional variables, then this truth table will have 2^100 entries.

there is also another way of simplifying theorem into a normal form: in a series of AND or OR

##### First Order Logic
First Order logic is based on propositional logic, but introduces:
1. Predicates, which are boolean functions
2. Quantifiers, which are "forall" or "there exists" and are basically reduce operations from [Bool] -> Bool

With this new instruments we are able to work with sets:
for example (forall x Loves(Me, x)) is true if every x loves Me
or (there exists x Loves(Me, x)) is true if atleast one x loves Me

##### Back to Turing
The main points of this argument is to show that first order logic <=> Turing machines.

Turing did show the systematic description of first order logic expressions: basically the way to encode any first order logic expression into Turing Machines's tape.

Then Turing introduces the machine which given the Description Number of the first order logic expression can proove it.
Since first order logic is Godel complete, then we can create any True theorems from the first order logic axioms. It means that:
- if we given the fol axioms and DN X of fol expression
- then machine can generate all possible theorems from the axioms
- it can iterate every theorem and check for equality with X
- if such theorem is found, then X is a theorem
- otherwise it will find it indefinetely

Which means that fol is semi-decidable: decidable if X is indeed a theorem, undecidable if X is not a theorem

##### Section 11. Giving examples of large classes of numbers which are computable
In this section Turing illustrates that his Turing Machine can compute many familiar numbers:
1. rational numbers
2. algebraic numbers: root of polynomials with rational coefficients
3. numbers defined by convergent power series (e, pi, sin, ...)
4. 

In [107]:
def bisection(f, a, b, N):
    fa = f(a)
    fb = f(b)
    assert fa * fb < 0
    while not (abs(fa) < N or abs(fb) < N):
        c = (a + b) / 2
        fc = f(c)
        if fa * fc < 0:
            b = c
        else:
            a = c
        fa = f(a)
        fb = f(b)
    return a if abs(fa) < abs(fb) else b       

bisection(lambda x: 5 * x**3 - 3 * x**2 - 7, -0.5, 1.5, 0.0001)

1.3585433959960938

### 14. The Major Proof
> In Section 11, Turing shows how the functionality of a computing machine can be expressed in the language of first-order predicate logic. He then constructs a formula in this logic that is provable if and only if the machine ever prints the digit 0

> If that formula is decidable - that is, if we can determine whether it's provable - then we'd have a general process for determining whether a machine ever prints 0, and we already know we can't have one.

That's said, Turing will actualy use the Halting Problem to proove that there is no general algorithm for decidability.

**K** is for restricted functional calculus (first order predicate logic)

Turing proposes that there is no general process of determining whether a given formula U of the K is provable

##### Godel Incompletness and Turing Undecidability
- Godel did proove that for any system with arithmetic axioms such system is incomplete: there exist formula U and U and -U cannot be prooved e.g. we cannot proove every True theorems in such system
- isn't it also imply the undecidability of such system? meaning there is no general process for determining whether statement U is proovable or not
- well no, such process can still exist and just determine not proovability of U and -U
- so Turing will proove than no such general process exist

The plan for the proof is simple:
- introduce function Un which takes computing machine M as input (via description number)
- in other words, convert Turing Machine's description (M) into language of first order logic
- show that if there is a general process for determining proovability of Un(M)
- then there is a general process of determing whether machine M will print 0
- but we know that no such process exist (halting problem), hence no such general process of provability exist

As we know, Turing Machine's description number is basically instructions. This instructions then passed into Universal Machine and are executed.

Instructions are executed one by one. After each step of execution we got a new complete configuration (machine's snapshot).

To convert Turing Machine into language of first order logic we need to convert each instruction into fol.

Complete configuration is an array of symbols:
```raw
[everything left from scanned symbol] + [state] + [scanned symbol] + [everithing right from scanned symbol]
```

##### Universal Turing Machine in Python

In [171]:
def complete_configuration(tape, head, state):
    return tape[0:head] + [state] + [tape[head]] + tape[(head + 1):]

# move = 0 => no move
# move = 1 => move left
# move = 2 => move right
def instruction(state, symbol, symbol2print, move, next_state):
    return [state, symbol, symbol2print, move, next_state]

tape = instruction('q1', 0, 1, 2, 'q2') + instruction('q2', 0, 2, 2, 'q3') + instruction('q3', 0, 3, 2, 'q2') + [-1] + complete_configuration([0 for x in range(32)], 0, 'q1')

def step():
    global tape
    last_minus1_index = - 1
    for i, x in enumerate(tape):
        if x == -1:
            last_minus1_index = i
    assert last_minus1_index != -1
    last_complete_config = tape[last_minus1_index + 1:]
    state_index = -1
    for i, x in enumerate(last_complete_config):
        if isinstance(x, str) and x.startswith('q'):
            state_index = i
            break
    state = last_complete_config[state_index]
    symbol = last_complete_config[state_index + 1]
    instr_index = -1
    for i, x in enumerate(tape):
        if x == state and tape[i + 1] == symbol:
            instr_index = i
            break
    assert instr_index != -1
    to_print = tape[instr_index + 2]
    move = tape[instr_index + 3]
    next_state = tape[instr_index + 4]
    if to_print != symbol:
        print(to_print)
    next_complete_config = []
    left_symbol = None
    if state_index > 0:
        next_complete_config += [last_complete_config[0:state_index - 1]]
        left_symbol = last_complete_config[state_index - 1]
    if move == 0: # no move
        if left_symbol is not None:
            next_complete_config += [left_symbol]
        next_complete_config += [next_state]
        next_complete_config += [to_print]
    elif move == 1: # left
        next_complete_config += [next_state]
        if left_symbol is not None:
            next_complete_config += [left_symbol]
        next_complete_config += [to_print]
    elif move == 2: # right
        if left_symbol is not None:
            next_complete_config += [left_symbol]
        next_complete_config += [to_print]
        next_complete_config += [next_state]
    next_complete_config += last_complete_config[state_index + 2:]
    tape += [-1] + next_complete_config

for i in range(16):
    step()


1
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2


R_S(x, y) - in the complete configuration x the symbol on the square y is S
I(x, y) - in the complete configuration x the square y is scanned (head posiiton is at 6th square of the tape)
K_q(x) - in the complete configuration x the state is q
F(x, y) - y = x + 1

if (configs[x].tape[y] == Sj && configs[x].head == y && configs[x].state == q_i && x' = x + 1 && y' = y + 1) then
x' is index of next complete config
y' is index of 
()


### Lambda Calculus
1 = { a, b -> a(b) }
S = { p, f, x -> f(p(f, x)) }

S(1) = { p, f, x -> f(p(f, x))}(1)
S(1) = { p, f, x -> f(p(f, x))}({a, b -> a(b)})
S(1) = { f, x -> f({a, b -> a(b)}(f, x))}
S(1) = { f, x -> f(f(x))}

below is python implementation of lambda calculus definitions of:
- one as 1
- S as succesor to define all other natural numbers
- plus, mul operations on two natural numbers
- illustration of power 2^3

In [ ]:
one = lambda a: lambda b: a(b)
S = lambda p: lambda f: lambda x: f(p(f)(x))

eval = lambda x: x(lambda x: x + 1)(0)

eval(S(S(S(one))))

plus = lambda p: lambda o: lambda f: lambda x: p(f)(o(f)(x))
mul = lambda p: lambda o: lambda x: p(o(x))

print(eval(plus(S(one))(S(S(one)))))
print(eval(mul(S(S(one)))(S(S(one)))))

print(eval(S(S(one))(S(one)))) # power

5
9
8


##### Turing's appendix about lambda calculus and computability
The main theorem: the sequence which is definable by lambda calculus is also computable

- there is a sequence y
- y[n] = phi_y(n) 
- sequence y is lambda-definable if 1 + phi_y(n) is a lambda-definable function of n

in other words:
- let N_n be a lambda-definalbe integer n (N_2 = S(1), N_3 = S(S(1)), ...)
- if there is lambda-definable M_y such that
- forall integers n {M_y}(N_n) conv N_(phi_y(n) + 1) or basically {M_y}(N_n) = phi_y(n) + 1

> this + 1 is necessary since in lamda calculus first natural number is 1, but in Turing's machine the first number is 0
> the y[n] is either 0 or 1, but to convert it to lambda-calculus it should be 1 or 2

that whole idea is basically:
1. if we have a sequence y which digits are defined by phi_y(n): y[n] = phi_y(n)
2. then y is lambda-definable if phi_y(n) + 1 is lambda definable

since {M_y}(N_n) is either 1 or 2 we can:
- create a machine L_2 which is supplied with {M_y}(N_n)
- this machine then perform transformations to supplied formula
- each of transformation result in another formula (another form of initial {M_y}(N_n))
- eventually L_2 will find that {M_y}(N_n) is either N_1 or N_2 (1 or 2)


Now Turing wants to prove that every computable sequence y is lambda-definable. For this he wants to show how to find formula M_y such that for all integers n:
```raw
{M_y}(N_n) conv N[1 + phi_y(n)]
```

So we have a machine M that computes sequence y.
Then let e(n) be the DN of n-th complete configuration of M (n is basically index of complete configuration)

It's obvious that there is a relation between n-th and n+1th complete configuration of the machine:
- to produce next configuration we rely only on the previous configuration. that's clear if you understand Universal Machine
- so there is e(n + 1) = p_y(e(n)) where p_y is determined by Instructions of M (and this Instructions basically define sequence y)
- the p_y function is lambda-definable. in other words there is lambda-definable A_y such that for all integers n `{A_y}(N[e(n)]) conv N[e(n+1)]`
- let `U = \u -> {{u}(A_y)}(N[e(0)])` where N[e(0)] is lambda-defined description number of initial (0th) complete configuration
```raw
U = \u -> {{u}(A_y)}(N[e(0)])
{U}(N[n]) = {{N[n]}(A_y)}(N[e(0)])
N[n] = \f x -> (call f n times)(x)
{U}(N[n]) = {{\f x -> (call f n times)(x)}(A_y)}(N[e(0)])
{U}(N[n]) = {\x -> (call A_y n times)(x)}(N[e(0)])
{U}(N[n]) = (call A_y n times)N[e(0)]
{U}(N[n]) = (n times of A_y) ... A_y . A_y . A_y N[e(0)] # f composition like in haskell
{U}(N[n]) = N[e(n)]
```

then Turing states that there exist lambda-definable formula V such that:
```raw
{{V}(N[e(n + 1)])}(N[e(n)]) = 
    if figure 0 printed betwee n and n+1 cc {
        conv N[1]
    } else if 1 printed {
        conv N[2]
    } else {
        conv N[3]
    }
```

and such formula should really exist because to execute n-th step of Turing machine all we need is nth cc and instructions or nth and n+1th cc. we can even compare the tape between this ccs and find the difference. the diff will be printed symbols

then W_y and Q is introduced where:
```raw
{W_y}(N[n]) = {{V}(N[e(n + 1)])}(N[e(n)])
or basically the output of n-th step (N[1] or N[2] or N[3] since some steps don't print anything) 

{{Q}(W_y)}(N[s]) conv N[r(s)] 
where N[r(s)] is s-th integer for which {W_y}(N[s]) will be either N[1] or N[2]
so if we do something like [{{Q}(W_y)}(N[x]) for x in range(10)] then we will get first 10 complete configuration indices for which N[1] or N[2] were printed

and finally:
M_y = \w -> {W_y}({{Q}(W_y)}(w))

M_y is lamda-defined function that computes y computable sequence

[{M_y}(i) for i in range(10)] will get us first 10 digits of y sequence
```

In [ ]:
# O(N^2) - polynomial time complexity
def contain(xs, ys):
    for x in xs:
        con = False
        for y in ys:
            if x == y:
                con = True
                break
        if not con:
            return False
    return True

# O(2^N) - exponential time
def all_permutations(n, size):
    word = [0] * n
    i = 0
    yield word
    while i < n:
        if word[i] < size:
            word[i] += 1
            i = 0
            yield word
        else:
            word[i] = 0
            i += 1


# contain([1,2,123,3], [1,2,3,4,5,6])
gen = all_permutations(3, 2)
for x in gen:
    print(x)

[0, 0, 0]
[1, 0, 0]
[2, 0, 0]
[0, 1, 0]
[1, 1, 0]
[2, 1, 0]
[0, 2, 0]
[1, 2, 0]
[2, 2, 0]
[0, 0, 1]
[1, 0, 1]
[2, 0, 1]
[0, 1, 1]
[1, 1, 1]
[2, 1, 1]
[0, 2, 1]
[1, 2, 1]
[2, 2, 1]
[0, 0, 2]
[1, 0, 2]
[2, 0, 2]
[0, 1, 2]
[1, 1, 2]
[2, 1, 2]
[0, 2, 2]
[1, 2, 2]
[2, 2, 2]


In [233]:
list(range(0, 0))

[]